# ممیزی دادهٔ Divar و پایلوت Laya

## tl;dr

منبع یک‌میلیون‌ردیفی Divar پس از حذف موارد ناسازگار به ۹۹۹٬۴۱۶ ردیف fact رسید؛ این داده آگهی عرضه است، نه نیاز واقعی. پایلوت محلیِ مدل دقیق `convaiinnovations/laya-multilingual` تعداد ۳۸۹ ردیف shadow تولید کرد و همگی صریحاً برای آموزش غیرمجاز هستند. این دفترچه شمارش‌ها و توافق محدود با facts آگهی را بازتولید می‌کند؛ دقت روی نیازهای کاربران را اندازه نمی‌گیرد.

آموزش انجام نشده است.

## Context & Methods

### Key Assumptions

- Grain منبع: یک ردیفِ عرضهٔ ملکی با متن فروشنده/مشاور و facts ساختاریافته.
- Grain پایلوت: یک proposal از Laya برای یک ردیف عرضه؛ هیچ پیش‌بینی مدل gold label نیست.
- توافق با `offer_*` فقط proxy است و به‌علت تفاوت perspective، benchmark نیاز کاربر نیست.
- شمارش‌ها بدون چاپ متن خام انجام می‌شود؛ تمام ورودی‌ها از فایل‌های محلی زیر خوانده می‌شوند.

فایل منبع پردازش‌شده: `data/divar/divar-property-offer-facts-v2.jsonl` و manifest مجاور آن. فایل پایلوت: `data/divar/divar-laya-shadow-diverse-pilot-v1.jsonl` و manifest مجاور آن. روش official Laya از typed decisions استفاده می‌کند (`choice`, `score`, `noul`) و متن تولید نمی‌کند؛ notebook آموزش رسمی از checkpoint دیگری آغاز می‌شود، پس عیناً برای این کار قابل استفاده نیست. [مخزن Laya](https://github.com/NandhaKishorM/laya) · [notebook رسمی fine-tune](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb)

In [ ]:
from collections import Counter
from pathlib import Path
import json

ROOT = Path.cwd().resolve()
while not (ROOT / 'package.json').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('Run this notebook from inside the NiazFinder checkout.')
    ROOT = ROOT.parent

CORPUS_PATH = ROOT / 'data/divar/divar-property-offer-facts-v2.jsonl'
PILOT_PATH = ROOT / 'data/divar/divar-laya-shadow-diverse-pilot-v1.jsonl'
CORPUS_MANIFEST_PATH = Path(str(CORPUS_PATH) + '.manifest.json')
PILOT_MANIFEST_PATH = Path(str(PILOT_PATH) + '.manifest.json')

def iter_jsonl(path):
    with path.open('r', encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, start=1):
            if line.strip():
                try:
                    yield json.loads(line)
                except json.JSONDecodeError as error:
                    raise ValueError(f'Malformed JSONL at line {line_number}') from error

assert CORPUS_PATH.is_file() and CORPUS_MANIFEST_PATH.is_file()
assert PILOT_PATH.is_file() and PILOT_MANIFEST_PATH.is_file()
corpus_manifest = json.loads(CORPUS_MANIFEST_PATH.read_text(encoding='utf-8'))
pilot_manifest = json.loads(PILOT_MANIFEST_PATH.read_text(encoding='utf-8'))
print({'source_rows_in_manifest': corpus_manifest.get('outputRows'), 'pilot_rows_in_manifest': pilot_manifest.get('rowsProcessed'), 'model': pilot_manifest.get('model')})

## Data

### 1. Profile the normalized source

Stream the full JSONL without loading raw text into the output. City and neighborhood coverage is measured against the app mapping fields in each row.

In [ ]:
row_count = 0
source_rows_with_text = 0
app_city_rows = 0
app_neighborhood_rows = 0
city_slugs = set()
neighborhood_ids = set()
category_counts = Counter()

for row in iter_jsonl(CORPUS_PATH):
    row_count += 1
    source_rows_with_text += int(bool(row.get('state', '').strip()))
    location = row.get('offerLocation') or {}
    city_slug = location.get('appCitySlug')
    neighborhood_id = location.get('appNeighborhoodId')
    if city_slug:
        app_city_rows += 1
        city_slugs.add(city_slug)
    if neighborhood_id:
        app_neighborhood_rows += 1
        neighborhood_ids.add(neighborhood_id)
    category = ((row.get('typedDecisions') or {}).get('offer_category') or {}).get('value')
    if category:
        category_counts[category] += 1

source_profile = {
    'rows': row_count,
    'rows_with_text': source_rows_with_text,
    'app_city_mapped_rows': app_city_rows,
    'app_city_mapping_rate': round(app_city_rows / row_count, 4),
    'distinct_app_cities': len(city_slugs),
    'app_neighborhood_mapped_rows': app_neighborhood_rows,
    'app_neighborhood_mapping_rate': round(app_neighborhood_rows / row_count, 4),
    'distinct_app_neighborhood_ids': len(neighborhood_ids),
    'distinct_offer_categories': len(category_counts),
    'offer_category_counts': dict(sorted(category_counts.items())),
}
print(json.dumps(source_profile, ensure_ascii=False, indent=2))

## Results

### 2. Validate the shadow sample and field-level behavior

`unknown` is a legitimate abstention and is reported separately from non-unknown proposals. Exact matches below compare model choices to seller/agent offer metadata only.

In [ ]:
pilot_rows = 0
source_ids = set()
devices = Counter()
categories = set()
cities = set()
answer_counts = Counter()
unknown_counts = Counter()
predicted_counts = Counter()
source_fact_metrics = {
    'offer_category': {'known': 0, 'predicted': 0, 'exact': 0},
    'offer_property_kind': {'known': 0, 'predicted': 0, 'exact': 0},
    'offer_transaction_type': {'known': 0, 'predicted': 0, 'exact': 0},
}
answer_key_for_fact = {
    'offer_category': 'category_candidate',
    'offer_property_kind': 'property_kind',
    'offer_transaction_type': 'transaction_type',
}

for row in iter_jsonl(PILOT_PATH):
    pilot_rows += 1
    assert row.get('synthetic') is True
    assert row.get('derivedFromSupplyListing') is True
    assert row.get('isNeedGroundTruth') is False
    assert row.get('trainingEligible') is False
    assert row.get('shadowOnly') is True
    assert row.get('laya', {}).get('model') == 'convaiinnovations/laya-multilingual'
    source_ids.add((row.get('source') or {}).get('sourceExampleId'))
    devices[row.get('laya', {}).get('device', 'unknown')] += 1
    location = row.get('sourceOfferLocation') or {}
    if location.get('appCitySlug'):
        cities.add(location['appCitySlug'])
    category = ((row.get('sourceOfferFacts') or {}).get('offer_category') or {}).get('value')
    if category:
        categories.add(category)
    answers = (row.get('laya') or {}).get('answers') or {}
    facts = row.get('sourceOfferFacts') or {}
    for field, answer in answers.items():
        choice = answer.get('choice', 'unknown') if isinstance(answer, dict) else 'unknown'
        answer_counts[field] += 1
        if choice == 'unknown':
            unknown_counts[field] += 1
        else:
            predicted_counts[field] += 1
    for fact_key, answer_key in answer_key_for_fact.items():
        fact = (facts.get(fact_key) or {}).get('value')
        if not fact or fact == 'unknown':
            continue
        source_fact_metrics[fact_key]['known'] += 1
        choice = ((answers.get(answer_key) or {}).get('choice'))
        if choice and choice != 'unknown':
            source_fact_metrics[fact_key]['predicted'] += 1
            source_fact_metrics[fact_key]['exact'] += int(choice == fact)

pilot_profile = {
    'rows': pilot_rows,
    'distinct_source_examples': len(source_ids),
    'distinct_mapped_cities': len(cities),
    'distinct_offer_categories': len(categories),
    'devices': dict(devices),
    'field_unknown_rates': {
        field: round(unknown_counts[field] / count, 4)
        for field, count in answer_counts.items() if count
    },
    'offer_side_proxy_concordance': {
        field: {**values, 'agreement_among_non_unknown': round(values['exact'] / values['predicted'], 4) if values['predicted'] else None}
        for field, values in source_fact_metrics.items()
    },
}
print(json.dumps(pilot_profile, ensure_ascii=False, indent=2))

## Takeaways

| Check | Observed result |
|---|---:|
| Normalized source rows | 999,416 |
| Rows with app city mapping | 925,129 (92.56%) |
| Rows with exact app neighborhood mapping | 420,989 (42.12%) |
| Mapped app cities / property categories | 382 / 16 of 18 |
| Laya shadow rows | 389; all `trainingEligible: false` |
| Devices in pilot | 136 CPU, 253 MPS |
| `unknown` rate: category / transaction / property kind | 85.5% / 57.8% / 63.4% |
| Offer-side exact proxy agreement: category / transaction / property kind | 14.3% / 17.6% / 78.8% among non-unknown predictions |

- The pilot is broad in city/category variety but shallow for local-neighborhood ground truth; do not infer location quality from its Laya scores.
- Property-kind proxy agreement is stronger than transaction/category agreement, but still measures a seller-offer task and cannot validate seeker intent.
- Amenity statements in an offer describe the offered property, not what a seeker wants.
- None of the pilot rows is training eligible. Before fitting `/post`, obtain licensed/consented need-perspective records and independently reviewed typed-decision targets, then split by normalized-text group before training/calibration/test.
- The official Laya notebook demonstrates typed decision-head training, soft targets, proper-score RL, held-out calibration and test evaluation, but it trains `convaiinnovations/laya` rather than the only permitted multilingual checkpoint. An exact-checkpoint adaptation still needs a clean eligible dataset and a verified MPS-compatible training pass.

### Validation note

The notebook is valid nbformat 4 JSON and all four Python code cells were executed sequentially with the project Python 3.12.10 interpreter. `nbformat`, `nbclient`, and a Jupyter executable are not installed here, so native notebook rendering and saved cell outputs could not be produced in this environment.

In [ ]:
assert row_count == corpus_manifest.get('outputRows')
assert pilot_rows == pilot_manifest.get('rowsProcessed')
assert pilot_profile['distinct_source_examples'] == pilot_rows
assert set(devices) == set(pilot_manifest.get('devicesUsed', []))
assert all(row_count_for_field >= 0 for row_count_for_field in answer_counts.values())
print('Integrity gates passed; this does not qualify the pilot for training.')

## Live v5 bounded audit

این سلول فقط تعداد ردیف‌های commit‌شده در manifest را می‌خواند و همان پیشوند را با evaluator اجرا می‌کند؛ انتهای در حال نوشتن JSONL وارد ارزیابی نمی‌شود. نتایج، توافق با برچسب‌های فرضی مشتق‌شده از آگهی عرضه هستند، نه دقت روی نیاز واقعی و نه gold labels.

In [ ]:
import subprocess

V5_PATH = ROOT / 'data/divar/divar-hypothetical-needs-laya-full-v5-2026-09-26.jsonl'
V5_MANIFEST_PATH = Path(str(V5_PATH) + '.manifest.json')
v5_manifest = json.loads(V5_MANIFEST_PATH.read_text(encoding='utf-8'))
committed_rows = int(v5_manifest['rowsProcessed'])
evaluator = ROOT / 'scripts/datasets/evaluate-divar-hypothetical.ts'
completed = subprocess.run(
    ['bun', '--conditions=react-server', str(evaluator), str(V5_PATH), '--max-rows', str(committed_rows)],
    cwd=ROOT, check=True, capture_output=True, text=True,
)
v5_evaluation = json.loads(completed.stdout)
assert v5_evaluation['rows'] == committed_rows
assert v5_evaluation['trainingEligibleRows'] == 0
category_metric = v5_evaluation['laya']['category_candidate']
category_predictions = v5_evaluation['categoryCounterfactualAgreement']['perExpectedCategory']
summary = {
    'scope': v5_evaluation['evaluationScope'],
    'rows': v5_evaluation['rows'],
    'label_status': v5_evaluation['labelStatus'],
    'training_eligible_rows': v5_evaluation['trainingEligibleRows'],
    'category_exact_counterfactual_agreement_pct': category_metric['exactMatchPct'],
    'category_top_confusions': v5_evaluation['categoryCounterfactualAgreement']['topConfusions'][:5],
    'category_counts': category_predictions,
    'transaction_exact_counterfactual_agreement_pct': v5_evaluation['laya']['transaction_type']['exactMatchPct'],
    'property_kind_exact_counterfactual_agreement_pct': v5_evaluation['laya']['property_kind']['exactMatchPct'],
    'full_catalog_city_resolver_consistency_pct': v5_evaluation['fullCatalogCityResolver']['exactMatchPct'],
    'catalog_inserted_neighborhood_resolver_consistency_pct': v5_evaluation['deterministicNeighborhood']['exactTopOnePct'],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

### 3. Recheck Divar coverage against current app catalogs

این audit کاملِ منبع را فقط تجمیعی می‌خواند تا پوشش شهرها/دسته‌ها/محله‌ها را از روی catalog فعلی پروژه گزارش کند؛ اختلاف آمار خام و نرمال‌شده را هم نشان می‌دهد.

In [ ]:
coverage_run = subprocess.run(
    ['bun', 'scripts/datasets/audit-divar-app-coverage.ts'],
    cwd=ROOT, check=True, capture_output=True, text=True,
)
coverage = json.loads(coverage_run.stdout)
assert coverage['sourceRowsRead'] == 1_000_000
assert coverage['normalizedOfferRows'] == 999_416
assert coverage['appLocations']['cityCatalogs'] == 1_207
assert coverage['sourceNeighborhoodCoverage']['shareOfNormalizedOffersWithMappedNeighborhoodPct'] == 42.12
print(json.dumps({
    'rows': coverage['sourceRowReconciliation'],
    'categories': coverage['appCategories'],
    'app_locations': coverage['appLocations'],
    'source_city_coverage': coverage['sourceCityCoverage'],
    'source_neighborhood_coverage': coverage['sourceNeighborhoodCoverage'],
    'rights': coverage['transferAndRights'],
}, ensure_ascii=False, indent=2))

### 4. Local category-question wording A/B

این آزمون ۱۹ عبارت دست‌نویس را با گزینه‌های یکسان، یک بار با criteria قدیمیِ manifest و یک بار با توضیح معنایی جدید می‌سنجد. عبارت‌ها فقط diagnostic هستند؛ نه نیاز واقعیِ جمع‌آوری‌شده‌اند و نه دادهٔ آموزش. اجرای سلول به worker محلی `convaiinnovations/laya-multilingual` نیاز دارد و دو inference batch اجرا می‌کند.

In [ ]:
ab_run = subprocess.run(
    ['bun', 'scripts/datasets/benchmark-laya-category-prompts.ts'],
    cwd=ROOT, check=True, capture_output=True, text=True,
)
ab_results = json.loads(ab_run.stdout)
assert ab_results['fixtureStatus'] == 'authored-diagnostic-only-not-training-data'
assert ab_results['cases'] == 19
print(json.dumps({
    'fixtureStatus': ab_results['fixtureStatus'],
    'cases': ab_results['cases'],
    'baseline': ab_results['baselineExactAgreement'],
    'descriptive': ab_results['descriptiveExactAgreement'],
    'unknownCase': ab_results['unknownCase'],
}, ensure_ascii=False, indent=2))

## آخرین وضعیت اعتبارسنجی دفترچه

فایل nbformat و syntax همهٔ code cellها بررسی شده‌اند. evaluator محدود، ممیزی پوشش و اسکریپت A/B محلی جداگانه اجرا و بررسی شدند؛ خروجی‌های اجرای آن‌ها عمداً در notebook ذخیره نشده است. محیط Jupyter/nbclient نصب نیست، پس اجرای ترتیبی و render کامل خود notebook ادعا نمی‌شود.